In [79]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata
import urllib.request
import urllib
import pandas as pd
import sys


# # 검색 키워드 URL 변환
# import urllib.parse

# # 날짜 및 시간
# from datetime import datetime

In [80]:
# Step 2. 사용자에게 수집 조건을 입력받습니다.

print("=" * 80)
print("유튜브 숏츠 댓글 수집 프로그램")
print("=" * 80)

# 1. 수집 방식 선택
print("\n[수집 방식 선택]")
print("1. URL을 직접 입력하여 댓글 수집")
print("2. 키워드로 영상 검색 후 댓글 수집")

collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

while collect_type not in ['1', '2']:
    print("입력 오류입니다. 1 또는 2만 입력하세요.")
    collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

# 2. URL 또는 키워드 입력
if collect_type == '1':
    target_url = input("\n댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()
    search_keyword = ""
else:
    search_keyword = input("\n검색할 키워드를 입력하세요: ").strip()
    target_url = ""

# 3. 수집할 댓글 수 입력
try:
    comment_cnt = int(input("\n수집할 댓글 수를 입력하세요(기본값: 30): ").strip())
except ValueError:
    comment_cnt = 30
    print("숫자가 입력되지 않아 기본값 30건으로 설정합니다.")

if comment_cnt <= 0:
    comment_cnt = 30
    print("0 이하의 값은 입력할 수 없어 기본값 30건으로 설정합니다.")

# 4. 저장 폴더명 입력
save_folder = input("\n저장할 폴더명을 입력하세요(기본값:c:\\py_temp\\): ").strip()

if save_folder == "":
    save_folder = "c:\\py_temp\\"

# 5. 입력 결과 확인
print("\n" + "=" * 80)
print("입력한 수집 조건")
print("=" * 80)

if collect_type == '1':
    print("수집 방식   : URL 직접 입력")
    print("대상 URL    :", target_url)
else:
    print("수집 방식   : 키워드 검색")
    print("검색 키워드 :", search_keyword)

print("수집 댓글 수 :", comment_cnt)
print("저장 폴더명  :", save_folder)
print("=" * 80)

유튜브 숏츠 댓글 수집 프로그램

[수집 방식 선택]
1. URL을 직접 입력하여 댓글 수집
2. 키워드로 영상 검색 후 댓글 수집
입력 오류입니다. 1 또는 2만 입력하세요.
숫자가 입력되지 않아 기본값 30건으로 설정합니다.

입력한 수집 조건
수집 방식   : URL 직접 입력
대상 URL    : https://www.youtube.com/shorts/UkU_P-jfBtM
수집 댓글 수 : 30
저장 폴더명  : c:\py_temp\


In [81]:
# # 더 안전하게 쓰는 코드 
# # 2. URL 또는 키워드 입력
# if collect_type == '1':
#     target_url = input("\n댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()
    
#     while target_url == "":
#         print("URL이 비어 있습니다. 다시 입력하세요.")
#         target_url = input("댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()

#     if not target_url.startswith("http"):
#         print("URL 앞에 https:// 가 없어 자동으로 붙입니다.")
#         target_url = "https://" + target_url

#     search_keyword = ""

# else:
#     search_keyword = input("\n검색할 키워드를 입력하세요: ").strip()
#     target_url = ""

In [82]:
#Step 3. 크롬 드라이버 설정 및 웹 페이지 열기 ~ 일단 유튜브에 들어가자 
s = Service("c:/py_temp/chromedriver.exe") 
driver = webdriver.Chrome(service=s) 

url = 'https://www.youtube.com' 
driver.get(url) 
driver.maximize_window() 
time.sleep(3) 



In [83]:
# Step 4. 선택한 방식에 따라 실제 페이지로 이동하기

if collect_type == '1':
    # URL 직접 입력 방식
    driver.get(target_url)
    time.sleep(3)
    print("\n입력한 숏츠 URL로 이동했습니다.")

elif collect_type == '2':
    # 키워드 검색 방식
    search_box = driver.find_element(By.NAME, "search_query")
    search_box.clear()
    search_box.send_keys(search_keyword)
    search_box.send_keys(Keys.ENTER)
    time.sleep(3)
    print("\n키워드 검색 결과 페이지로 이동했습니다.")


입력한 숏츠 URL로 이동했습니다.


In [84]:
print("collect_type :", collect_type)
print("target_url   :", target_url)
print("search_keyword :", search_keyword)

collect_type : 1
target_url   : https://www.youtube.com/shorts/UkU_P-jfBtM
search_keyword : 


In [85]:
# Step 5. 동영상 제목 가져오기

try:
    video_title = driver.find_element(
        By.CSS_SELECTOR,
        "yt-shorts-video-title-view-model h2 span"
    ).text

    video_title = video_title.split("#")[0].strip()

    print("동영상 제목 :", video_title)

except:
    video_title = "제목 없음"
    print("동영상 제목을 가져오지 못했습니다.")

video_url = driver.current_url

video_no = 1


동영상 제목 : 올리브영 알바생이 추천하는 올영 추천템 5가지


In [86]:
comment_btn = driver.find_element(
    By.CSS_SELECTOR,
    'button[aria-label*="댓글"]'
)

comment_btn.click()

In [87]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time

# Step 7. 댓글창 영역으로 마우스 이동 후 포커스 주기
try:
    comment_area = driver.find_element(
        By.CSS_SELECTOR,
        'ytd-comments[engagement-panel] #contents'
    )
except:
    comment_area = driver.find_element(
        By.CSS_SELECTOR,
        'ytd-comments[engagement-panel] ytd-comment-thread-renderer'
    )

ActionChains(driver).move_to_element(comment_area).click().perform()
time.sleep(1)

print("댓글창에 포커스를 주었습니다.")

댓글창에 포커스를 주었습니다.
